# CCE PoC v3: multi-model + semantic-entropy ablation

Runs the full v3 PoC on **all three LLMs in a single notebook execution**, then aggregates. Each model is launched as a separate subprocess so VRAM is freed cleanly between runs.

**What v3 adds over v2:**
- 3 models (CodeLlama-7B, Qwen2.5-Coder-7B, DeepSeek-Coder-7B)
- Semantic entropy (Kuhn 2023 / Farquhar 2024) as a 7th feature group
- 3 new ablation arms: `semantic_entropy_only`, `flare_plus_se`, `all_features`

**Total wall-clock on A100:** ~4.5 hours (3 models × 90 min each).

**Resumable:** if Colab disconnects mid-run, just open the notebook again and run all cells. Each model checks for cached features/SE/phase3 on Drive and skips what's done.

In [1]:
# All three models, run sequentially in this notebook.
# Each is a separate subprocess (`!python ...`) so VRAM is freed between runs.
# Total compute on A100: ~4.5 hours. On disconnect, just re-run — Drive cache resumes.

MODELS = [
    "Qwen/Qwen2.5-Coder-7B-Instruct",                # run 1: not gated, smallest risk
    "deepseek-ai/deepseek-coder-7b-instruct-v1.5",   # run 2: not gated
    "codellama/CodeLlama-7b-Instruct-hf",            # run 3: gated; license already accepted from v2
]
print(f'will run {len(MODELS)} models sequentially')

will run 3 models sequentially


## 2. Install + clone

In [2]:
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.8 MB/s eta 0:00:00


In [3]:
import os
if not os.path.exists('/content/reposynth'):
    !git clone https://github.com/aniJani/reposynth.git /content/reposynth
%cd /content/reposynth
!git checkout Research
!git pull --rebase || true

Cloning into '/content/reposynth'...
remote: Enumerating objects: 1213, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 1213 (delta 17), reused 23 (delta 9), pack-reused 1162 (from 1)
Receiving objects: 100% (1213/1213), 38.30 MiB | 19.44 MiB/s, done.
Resolving deltas: 100% (610/610), done.
/content/reposynth
Branch 'Research' set up to track remote branch 'Research' from 'origin'.
Switched to a new branch 'Research'
Already up to date.


## 3. HuggingFace login (CodeLlama and DeepSeek are gated; Qwen is not)

In [4]:
from huggingface_hub import login
from google.colab import userdata
try:
    login(userdata.get('HF_TOKEN'))
except Exception as e:
    print('Set HF_TOKEN as a Colab secret first:', e)

## 4. Mount Drive (per-model caches survive disconnects)

In [5]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/cce_poc_v3'
!mkdir -p {OUT_DIR}
print(f'will write per-model results to {OUT_DIR}')

Mounted at /content/drive
will write per-model results to /content/drive/MyDrive/cce_poc_v3


## 5. Run all three models sequentially

Each model is a separate subprocess — clean VRAM, isolated failures. If one model fails, the others still run; you can rerun this cell to retry.

In [6]:
import time, subprocess

%cd /content/reposynth

for i, model in enumerate(MODELS, 1):
    print(f"\n{'='*70}\n[{i}/{len(MODELS)}] running {model}\n{'='*70}", flush=True)
    t0 = time.time()
    rc = subprocess.call([
        "python", "/content/reposynth/research/paper/cce_poc_v3.py",
        "--model", model,
        "--repos-dir", "/content",
        "--out-dir", OUT_DIR,
    ])
    dt = (time.time() - t0) / 60
    if rc == 0:
        print(f"\n[{i}/{len(MODELS)}] ✓ {model}  ({dt:.1f} min)", flush=True)
    else:
        print(f"\n[{i}/{len(MODELS)}] ✗ {model}  rc={rc}  ({dt:.1f} min)", flush=True)
        print("Continuing to next model. Re-run this cell later to retry failures.", flush=True)
print("\nAll models attempted. Inspect individual results below or skip to aggregation.")

/content/reposynth

[1/3] running Qwen/Qwen2.5-Coder-7B-Instruct

[1/3] ✓ Qwen/Qwen2.5-Coder-7B-Instruct  (109.3 min)

[2/3] running deepseek-ai/deepseek-coder-7b-instruct-v1.5

[2/3] ✓ deepseek-ai/deepseek-coder-7b-instruct-v1.5  (169.9 min)

[3/3] running codellama/CodeLlama-7b-Instruct-hf

[3/3] ✓ codellama/CodeLlama-7b-Instruct-hf  (132.4 min)

All models attempted. Inspect individual results below or skip to aggregation.


## 6. Inspect per-model results

In [7]:
import json, re, glob

for model in MODELS:
    slug = re.sub(r'[^A-Za-z0-9]+', '_', model).strip('_')
    path = f'{OUT_DIR}/results__{slug}.json'
    try:
        with open(path) as f:
            res = json.load(f)
    except FileNotFoundError:
        print(f"\n=== {model} ===  (results not found at {path})")
        continue
    print(f"\n=== {res['model']}  (n_tasks={res['n_tasks']}) ===")
    print(f"{'arm':<22} {'#feat':>5} {'acc':>5} {'prec':>5} {'rec':>5} {'f1':>5}")
    for arm, r in res['phase2_loo_ablation'].items():
        print(f"{arm:<22} {r['n_features']:>5d} {r['accuracy']:>5.3f} "
              f"{r['precision']:>5.3f} {r['recall']:>5.3f} {r['f1']:>5.3f}")
    if res.get('phase3_end_to_end'):
        print('Phase 3:')
        print(f"  {'arm':<22} {'final':>5} {'always':>5} {'used':>5} {'save%':>6}")
        for arm, r in res['phase3_end_to_end'].items():
            print(f"  {arm:<22} {r['final_accuracy']:>5.3f} {r['always_retrieve_accuracy']:>5.3f} "
                  f"{r['n_retrievals_used']:>5d} {r['retrieval_save_rate']*100:>5.1f}%")


=== Qwen/Qwen2.5-Coder-7B-Instruct  (n_tasks=80) ===
arm                    #feat   acc  prec   rec    f1
saplma_only                3 0.600 0.182 0.545 0.273
flare_only                 4 0.662 0.265 0.818 0.400
v6_full                   11 0.600 0.161 0.455 0.238
cce_only                   8 0.675 0.200 0.455 0.278
v6_plus_cce               19 0.688 0.231 0.545 0.324
drop_cce                  11 0.600 0.161 0.455 0.238
semantic_entropy_only      4 0.662 0.136 0.273 0.182
flare_plus_se              8 0.637 0.235 0.727 0.356
all_features              23 0.675 0.222 0.545 0.316
Phase 3:
  arm                    final always  used  save%
  saplma_only            0.900 0.963    33  58.8%
  flare_only             0.950 0.963    34  57.5%
  v6_full                0.900 0.963    31  61.3%
  cce_only               0.900 0.963    25  68.8%
  v6_plus_cce            0.900 0.963    26  67.5%
  drop_cce               0.900 0.963    31  61.3%
  semantic_entropy_only  0.887 0.963    22  72.5%
  flar

---
## 7. Aggregation across models *(run this cell ONCE after all three model runs complete)*

In [8]:
%cd /content/reposynth
!python /content/reposynth/research/paper/cce_poc_v3_aggregate.py \
    --in-dir {OUT_DIR} \
    --out    {OUT_DIR}/v3_combined_results.json

/content/reposynth

## Phase-2 F1 across models

arm | Qwen/Qwen2.5-Coder-7B-Instruct | codellama/CodeLlama-7b-Instruct-hf | deepseek-ai/deepseek-coder-7b-instruct-v1.5
--- | --- | --- | ---
all_features | 0.316 | 0.566 | 0.400
cce_only | 0.278 | 0.510 | 0.333
drop_cce | 0.238 | 0.630 | 0.408
flare_only | 0.400 | 0.654 | 0.360
flare_plus_se | 0.356 | 0.630 | 0.390
saplma_only | 0.273 | 0.508 | 0.159
semantic_entropy_only | 0.182 | 0.434 | 0.450
v6_full | 0.238 | 0.630 | 0.408
v6_plus_cce | 0.324 | 0.627 | 0.468

## Phase-3 final accuracy across models

arm | Qwen/Qwen2.5-Coder-7B-Instruct | codellama/CodeLlama-7b-Instruct-hf | deepseek-ai/deepseek-coder-7b-instruct-v1.5
--- | --- | --- | ---
all_features | 0.900 | 0.838 | 0.863
cce_only | 0.900 | 0.838 | 0.850
drop_cce | 0.900 | 0.850 | 0.825
flare_only | 0.950 | 0.850 | 0.863
flare_plus_se | 0.925 | 0.863 | 0.863
saplma_only | 0.900 | 0.838 | 0.800
semantic_entropy_only | 0.887 | 0.863 | 0.875
v6_full | 0.900 | 0.850 | 0.825
v6_plus_c

In [12]:
# === NLI-based semantic entropy validation (fp32, drop-in replacement) ===

import gc
import json
import math
import re
import time
from pathlib import Path

import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from transformers import AutoModelForSequenceClassification, AutoTokenizer

NLI_MODEL = "microsoft/deberta-large-mnli"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Ensure any previous broken model is gone
try:
    del nli_mdl, nli_tok
except NameError:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"[nli] loading {NLI_MODEL} on {DEVICE} (fp32)...", flush=True)

nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)

# DeBERTa attention is not fp16-safe; load in fp32 explicitly
nli_mdl = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL
).float().to(DEVICE).eval()


@torch.no_grad()
def nli_entail(premise: str, hypothesis: str) -> bool:
    inputs = nli_tok(
        premise,
        hypothesis,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(DEVICE)

    # Disable autocast so attention stays in fp32 throughout
    with torch.autocast(device_type="cuda", enabled=False):
        logits = nli_mdl(**inputs).logits[0]

    return int(torch.argmax(logits)) == 2  # 2 = entailment


def cluster_by_nli(gens):
    if not gens:
        return []

    clusters = [[0]]

    for i in range(1, len(gens)):
        gi = gens[i]
        placed = False

        for c in clusters:
            rep = gens[c[0]]

            if nli_entail(rep, gi) and nli_entail(gi, rep):
                c.append(i)
                placed = True
                break

        if not placed:
            clusters.append([i])

    return clusters


def nli_se_features(gens):
    n = len(gens)

    if n == 0:
        return {
            "nli_se": 0.0,
            "nli_se_norm": 0.0,
            "nli_n_clusters": 0,
            "nli_largest_cluster_frac": 0.0
        }

    clusters = cluster_by_nli(gens)
    sizes = [len(c) for c in clusters]
    probs = [s / n for s in sizes]

    h = -sum(p * math.log(p + 1e-12) for p in probs)
    norm = h / math.log(n) if n > 1 else 0.0

    return {
        "nli_se": float(h),
        "nli_se_norm": float(norm),
        "nli_n_clusters": int(len(clusters)),
        "nli_largest_cluster_frac": float(max(sizes) / n)
    }


def loo_f1(features_per_task, outcomes, feature_names, seed=42):
    y = np.array([0 if c else 1 for c in outcomes])
    n = len(y)

    X = np.array([
        [f.get(name, 0.0) for name in feature_names]
        for f in features_per_task
    ])

    preds = np.zeros(n, dtype=int)

    for i in range(n):
        mask = np.ones(n, dtype=bool)
        mask[i] = False

        if len(np.unique(y[mask])) < 2:
            continue

        clf = LogisticRegression(
            random_state=seed,
            max_iter=2000,
            class_weight="balanced"
        )

        clf.fit(X[mask], y[mask])
        preds[i] = int(clf.predict(X[i:i + 1])[0])

    tp = int(((preds == 1) & (y == 1)).sum())
    fp = int(((preds == 1) & (y == 0)).sum())
    fn = int(((preds == 0) & (y == 1)).sum())
    tn = int(((preds == 0) & (y == 0)).sum())

    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0

    return {
        "f1": f1,
        "accuracy": (tp + tn) / n if n else 0.0,
        "precision": p,
        "recall": r
    }


def slugify(s):
    return re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")


ARMS = {
    "flare_only": (
        "cce_mean",
        "cce_max",
        "cce_std",
        "cce_spikes"
    ),
    "embedding_se_only": (
        "semantic_entropy",
        "semantic_entropy_norm",
        "n_clusters",
        "largest_cluster_frac"
    ),
    "nli_se_only": (
        "nli_se",
        "nli_se_norm",
        "nli_n_clusters",
        "nli_largest_cluster_frac"
    ),
    "flare_plus_emb_se": (
        "cce_mean",
        "cce_max",
        "cce_std",
        "cce_spikes",
        "semantic_entropy",
        "semantic_entropy_norm",
        "n_clusters",
        "largest_cluster_frac"
    ),
    "flare_plus_nli_se": (
        "cce_mean",
        "cce_max",
        "cce_std",
        "cce_spikes",
        "nli_se",
        "nli_se_norm",
        "nli_n_clusters",
        "nli_largest_cluster_frac"
    ),
}

# Quick smoke test before the long loop
print("[nli] smoke test...", flush=True)

assert nli_entail(
    "the timeout is 5 seconds",
    "the default timeout is five seconds"
)

assert not nli_entail(
    "the timeout is 5 seconds",
    "the timeout is 50 seconds"
)

print("[nli] smoke test passed", flush=True)

summary = {}

for model in MODELS:
    slug = slugify(model)

    samples_path = Path(OUT_DIR) / f"se_samples__{slug}.json"
    features_path = Path(OUT_DIR) / f"features__{slug}.json"

    if not samples_path.exists() or not features_path.exists():
        print(f"\n[skip] {model}: cache files missing")
        continue

    print(f"\n=== {model} ===", flush=True)

    with open(samples_path) as f:
        se_blob = json.load(f)

    with open(features_path) as f:
        feat_blob = json.load(f)

    samples_per_task = se_blob["samples"]
    features_per_task = feat_blob["features"]
    initial_correct = feat_blob["initial_correct"]

    print(
        f"[nli] re-clustering {len(samples_per_task)} tasks via NLI...",
        flush=True
    )

    t0 = time.time()

    for i, samples in enumerate(samples_per_task):
        features_per_task[i].update(nli_se_features(samples))

        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(samples_per_task)} done", flush=True)

    print(f"[nli] {time.time() - t0:.1f}s", flush=True)

    emb_se = np.array([f["semantic_entropy"] for f in features_per_task])
    nli_se = np.array([f["nli_se"] for f in features_per_task])

    if emb_se.std() > 0 and nli_se.std() > 0:
        corr = float(np.corrcoef(emb_se, nli_se)[0, 1])
    else:
        corr = float("nan")

    arm_results = {
        a: loo_f1(features_per_task, initial_correct, feats)
        for a, feats in ARMS.items()
    }

    print(f"  Pearson(emb_se, nli_se): {corr:+.3f}")
    print(f"  {'arm':<22} {'f1':>5} {'acc':>5}")

    for a, r in arm_results.items():
        print(f"  {a:<22} {r['f1']:>5.3f} {r['accuracy']:>5.3f}")

    summary[model] = {
        "correlation": corr,
        "arms": arm_results
    }

    out_path = Path(OUT_DIR) / f"features_with_nli__{slug}.json"

    with open(out_path, "w") as f:
        json.dump(
            {
                "features": features_per_task,
                "initial_correct": initial_correct,
                "qids": feat_blob.get("qids", [])
            },
            f,
            indent=2
        )

print("\n" + "=" * 78)
print("CROSS-MODEL SUMMARY: embedding-based SE vs NLI-based SE")
print("=" * 78)

print(f"\n{'model':<55} {'corr':>7}")

for m, d in summary.items():
    print(f"{m:<55} {d['correlation']:>+7.3f}")

print(f"\n{'arm':<22}", end="")

for m in summary:
    print(f" {m.split('/')[-1][:22]:<22}", end="")

print()

for arm in ARMS:
    print(f"{arm:<22}", end="")

    for d in summary.values():
        print(f" f1={d['arms'][arm]['f1']:>5.3f}{'':<14}", end="")

    print()

print("\nInterpretation:")
print("  - corr > 0.85 + similar F1 → embedding approximation justified.")
print("  - low corr or different F1 → use nli_se_only as canonical SE.")

del nli_mdl, nli_tok
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n[done]")

[nli] loading microsoft/deberta-large-mnli on cuda (fp32)...


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-large-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[nli] smoke test...


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 71, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 48, in spawn_con

AssertionError: 

In [10]:
import gc
import torch
from transformers import AutoModelForSequenceClassification

# Free the broken fp16 model
try:
    del nli_mdl
except NameError:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Reload in fp32 (DeBERTa attention is not fp16-safe)
NLI_MODEL = "microsoft/deberta-large-mnli"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

nli_mdl = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL
).to(DEVICE).eval()

print(f"[nli] reloaded {NLI_MODEL} in fp32 on {DEVICE}")

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-large-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[nli] reloaded microsoft/deberta-large-mnli in fp32 on cuda


In [13]:
# === Continue from the failed smoke test ===
# nli_mdl, nli_tok, nli_entail, cluster_by_nli, nli_se_features, loo_f1,
# slugify, ARMS are all already in memory.

import json
import time
from pathlib import Path

import numpy as np

# Lighter smoke test: tautology must entail itself
print("[nli] tautology smoke test:")

print(
    "  ('cat is black', 'cat is black') →",
    nli_entail("the cat is black", "the cat is black")
)

print(
    "  ('cat is black', 'dog is white') →",
    nli_entail("the cat is black", "the dog is white")
)

# Don't assert; just print so we can see what NLI is doing.

summary = {}

for model in MODELS:
    slug = slugify(model)

    samples_path = Path(OUT_DIR) / f"se_samples__{slug}.json"
    features_path = Path(OUT_DIR) / f"features__{slug}.json"

    if not samples_path.exists() or not features_path.exists():
        print(f"\n[skip] {model}: cache files missing")
        continue

    print(f"\n=== {model} ===", flush=True)

    with open(samples_path) as f:
        se_blob = json.load(f)

    with open(features_path) as f:
        feat_blob = json.load(f)

    samples_per_task = se_blob["samples"]
    features_per_task = feat_blob["features"]
    initial_correct = feat_blob["initial_correct"]

    print(
        f"[nli] re-clustering {len(samples_per_task)} tasks...",
        flush=True
    )

    t0 = time.time()

    for i, samples in enumerate(samples_per_task):
        features_per_task[i].update(nli_se_features(samples))

        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(samples_per_task)} done", flush=True)

    print(f"[nli] {time.time() - t0:.1f}s", flush=True)

    emb_se = np.array([f["semantic_entropy"] for f in features_per_task])
    nli_se = np.array([f["nli_se"] for f in features_per_task])

    if emb_se.std() > 0 and nli_se.std() > 0:
        corr = float(np.corrcoef(emb_se, nli_se)[0, 1])
    else:
        corr = float("nan")

    arm_results = {
        a: loo_f1(features_per_task, initial_correct, feats)
        for a, feats in ARMS.items()
    }

    print(f"  Pearson(emb_se, nli_se): {corr:+.3f}")
    print(f"  {'arm':<22} {'f1':>5} {'acc':>5}")

    for a, r in arm_results.items():
        print(f"  {a:<22} {r['f1']:>5.3f} {r['accuracy']:>5.3f}")

    summary[model] = {
        "correlation": corr,
        "arms": arm_results
    }

    out_path = Path(OUT_DIR) / f"features_with_nli__{slug}.json"

    with open(out_path, "w") as f:
        json.dump(
            {
                "features": features_per_task,
                "initial_correct": initial_correct,
                "qids": feat_blob.get("qids", [])
            },
            f,
            indent=2
        )

print("\n" + "=" * 78)
print("CROSS-MODEL SUMMARY: embedding-based SE vs NLI-based SE")
print("=" * 78)

print(f"\n{'model':<55} {'corr':>7}")

for m, d in summary.items():
    print(f"{m:<55} {d['correlation']:>+7.3f}")

print(f"\n{'arm':<22}", end="")

for m in summary:
    print(f" {m.split('/')[-1][:22]:<22}", end="")

print()

for arm in ARMS:
    print(f"{arm:<22}", end="")

    for d in summary.values():
        print(f" f1={d['arms'][arm]['f1']:>5.3f}{'':<14}", end="")

    print()

print("\n[done]")

[nli] tautology smoke test:
  ('cat is black', 'cat is black') → True
  ('cat is black', 'dog is white') → False

=== Qwen/Qwen2.5-Coder-7B-Instruct ===
[nli] re-clustering 80 tasks...
  20/80 done
  40/80 done
  60/80 done
  80/80 done
[nli] 39.4s


KeyError: 'semantic_entropy'

In [14]:
# === Fix: merge embedding-SE from se_samples cache, then run the analysis ===

import json
import time
from pathlib import Path

import numpy as np

EMB_SE_KEYS = (
    "semantic_entropy",
    "semantic_entropy_norm",
    "n_clusters",
    "largest_cluster_frac"
)

summary = {}

for model in MODELS:
    slug = slugify(model)

    samples_path = Path(OUT_DIR) / f"se_samples__{slug}.json"
    features_path = Path(OUT_DIR) / f"features__{slug}.json"

    if not samples_path.exists() or not features_path.exists():
        print(f"\n[skip] {model}: cache files missing")
        continue

    print(f"\n=== {model} ===", flush=True)

    with open(samples_path) as f:
        se_blob = json.load(f)

    with open(features_path) as f:
        feat_blob = json.load(f)

    samples_per_task = se_blob["samples"]
    emb_se_per_task = se_blob.get("se_features", [])
    features_per_task = feat_blob["features"]
    initial_correct = feat_blob["initial_correct"]

    # Merge embedding-SE features into the Phase 1 features dict.
    if len(emb_se_per_task) != len(features_per_task):
        print(
            f"  [warn] SE cache length mismatch: "
            f"{len(emb_se_per_task)} vs {len(features_per_task)}"
        )

    for f, se in zip(features_per_task, emb_se_per_task):
        for k in EMB_SE_KEYS:
            if k in se:
                f[k] = se[k]

    print(f"[nli] re-clustering {len(samples_per_task)} tasks...", flush=True)

    t0 = time.time()

    for i, samples in enumerate(samples_per_task):
        features_per_task[i].update(nli_se_features(samples))

        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(samples_per_task)} done", flush=True)

    print(f"[nli] {time.time() - t0:.1f}s", flush=True)

    emb_se = np.array([
        f.get("semantic_entropy", 0.0)
        for f in features_per_task
    ])

    nli_se = np.array([f["nli_se"] for f in features_per_task])

    if emb_se.std() > 0 and nli_se.std() > 0:
        corr = float(np.corrcoef(emb_se, nli_se)[0, 1])
    else:
        corr = float("nan")

    arm_results = {
        a: loo_f1(features_per_task, initial_correct, feats)
        for a, feats in ARMS.items()
    }

    print(f"  Pearson(emb_se, nli_se): {corr:+.3f}")

    print(
        f"  emb_se range: [{emb_se.min():.3f}, {emb_se.max():.3f}]   "
        f"nli_se range: [{nli_se.min():.3f}, {nli_se.max():.3f}]"
    )

    print(f"  {'arm':<22} {'f1':>5} {'acc':>5}")

    for a, r in arm_results.items():
        print(f"  {a:<22} {r['f1']:>5.3f} {r['accuracy']:>5.3f}")

    summary[model] = {
        "correlation": corr,
        "arms": arm_results
    }

    out_path = Path(OUT_DIR) / f"features_with_nli__{slug}.json"

    with open(out_path, "w") as f:
        json.dump(
            {
                "features": features_per_task,
                "initial_correct": initial_correct,
                "qids": feat_blob.get("qids", [])
            },
            f,
            indent=2
        )

print("\n" + "=" * 78)
print("CROSS-MODEL SUMMARY: embedding-based SE vs NLI-based SE")
print("=" * 78)

print(f"\n{'model':<55} {'corr':>7}")

for m, d in summary.items():
    print(f"{m:<55} {d['correlation']:>+7.3f}")

print(f"\n{'arm':<22}", end="")

for m in summary:
    print(f" {m.split('/')[-1][:22]:<22}", end="")

print()

for arm in ARMS:
    print(f"{arm:<22}", end="")

    for d in summary.values():
        print(f" f1={d['arms'][arm]['f1']:>5.3f}{'':<14}", end="")

    print()

print("\n[done]")


=== Qwen/Qwen2.5-Coder-7B-Instruct ===
[nli] re-clustering 80 tasks...
  20/80 done
  40/80 done
  60/80 done
  80/80 done
[nli] 40.0s
  Pearson(emb_se, nli_se): +0.293
  emb_se range: [-0.000, 1.055]   nli_se range: [-0.000, 1.609]
  arm                       f1   acc
  flare_only             0.400 0.662
  embedding_se_only      0.182 0.662
  nli_se_only            0.516 0.812
  flare_plus_emb_se      0.356 0.637
  flare_plus_nli_se      0.500 0.800

=== deepseek-ai/deepseek-coder-7b-instruct-v1.5 ===
[nli] re-clustering 80 tasks...
  20/80 done
  40/80 done
  60/80 done
  80/80 done
[nli] 38.9s
  Pearson(emb_se, nli_se): +0.202
  emb_se range: [-0.000, 1.055]   nli_se range: [-0.000, 1.609]
  arm                       f1   acc
  flare_only             0.360 0.600
  embedding_se_only      0.450 0.725
  nli_se_only            0.500 0.725
  flare_plus_emb_se      0.390 0.688
  flare_plus_nli_se      0.375 0.625

=== codellama/CodeLlama-7b-Instruct-hf ===
[nli] re-clustering 80 tasks...

In [15]:
# === Phase 3 with NLI-SE features + bootstrap 95% CIs ===
# Pure local analysis — no GPU, no LLM load needed.
# Reads features_with_nli__*.json (just saved) and phase3__*.json (from v3 run).
# The Phase 3 cache contains an always-retrieve regeneration for every task,
# so we can compute Phase 3 numbers for any new arm by lookup, without re-generating.

import json
from pathlib import Path

import numpy as np
from sklearn.linear_model import LogisticRegression


def loo_predictions(features_per_task, outcomes, feature_names, seed=42):
    """LOO predictions for one feature subset. Returns (preds, probs)."""
    y = np.array([0 if c else 1 for c in outcomes])
    n = len(y)

    X = np.array([
        [f.get(name, 0.0) for name in feature_names]
        for f in features_per_task
    ])

    preds = np.zeros(n, dtype=int)
    probs = np.zeros(n, dtype=float)

    for i in range(n):
        mask = np.ones(n, dtype=bool)
        mask[i] = False

        if len(np.unique(y[mask])) < 2:
            continue

        clf = LogisticRegression(
            random_state=seed,
            max_iter=2000,
            class_weight="balanced"
        )

        clf.fit(X[mask], y[mask])

        preds[i] = int(clf.predict(X[i:i + 1])[0])
        probs[i] = float(clf.predict_proba(X[i:i + 1])[0][1])

    return preds, probs


def f1_from_pairs(pairs):
    preds = np.array([p for p, _ in pairs])
    gold = np.array([g for _, g in pairs])

    tp = int(((preds == 1) & (gold == 1)).sum())
    fp = int(((preds == 1) & (gold == 0)).sum())
    fn = int(((preds == 0) & (gold == 1)).sum())

    if tp == 0:
        return 0.0

    p = tp / (tp + fp)
    r = tp / (tp + fn)

    return float(2 * p * r / (p + r))


def bootstrap_ci(values, metric_fn, n_boot=1000, alpha=0.05, seed=42):
    """Returns (point, ci_low, ci_high) by percentile bootstrap."""
    rng = np.random.default_rng(seed)
    n = len(values)

    boot = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot.append(metric_fn([values[i] for i in idx]))

    boot = np.array(boot)

    return (
        float(metric_fn(values)),
        float(np.percentile(boot, 100 * alpha / 2)),
        float(np.percentile(boot, 100 * (1 - alpha / 2)))
    )


# Same ARMS as before but the SE arms now point at NLI features
ARMS_FINAL = {
    "no_signal_baseline": tuple(),
    "saplma_only": ("hs_mean_norm", "hs_std_norm", "hs_max_norm"),
    "flare_only": ("cce_mean", "cce_max", "cce_std", "cce_spikes"),
    "v6_full": (
        "hs_mean_norm", "hs_std_norm", "hs_max_norm",
        "cce_mean", "cce_max", "cce_std", "cce_spikes",
        "attn_mean", "attn_max", "attn_std", "response_length"
    ),
    "cce_only": (
        "real_cce_mean", "real_cce_max", "real_cce_std",
        "real_cce_spikes", "h_code_mean", "h_code_max",
        "h_lang_mean", "h_lang_max"
    ),
    "embedding_se_only": (
        "semantic_entropy",
        "semantic_entropy_norm",
        "n_clusters",
        "largest_cluster_frac"
    ),
    "nli_se_only": (
        "nli_se",
        "nli_se_norm",
        "nli_n_clusters",
        "nli_largest_cluster_frac"
    ),
    "flare_plus_emb_se": (
        "cce_mean", "cce_max", "cce_std", "cce_spikes",
        "semantic_entropy", "semantic_entropy_norm",
        "n_clusters", "largest_cluster_frac"
    ),
    "flare_plus_nli_se": (
        "cce_mean", "cce_max", "cce_std", "cce_spikes",
        "nli_se", "nli_se_norm",
        "nli_n_clusters", "nli_largest_cluster_frac"
    ),
}


def slugify(s):
    import re
    return re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")


final_summary = {}

for model in MODELS:
    slug = slugify(model)

    nli_path = Path(OUT_DIR) / f"features_with_nli__{slug}.json"
    p3_path = Path(OUT_DIR) / f"phase3__{slug}.json"

    if not nli_path.exists() or not p3_path.exists():
        print(f"\n[skip] {model}: missing {nli_path.name} or {p3_path.name}")
        continue

    with open(nli_path) as f:
        nli_blob = json.load(f)

    with open(p3_path) as f:
        p3_cache = json.load(f)

    features_per_task = nli_blob["features"]
    initial_correct = nli_blob["initial_correct"]
    qids = nli_blob.get("qids", list(range(len(features_per_task))))
    n = len(features_per_task)

    # Look up always-retrieve outcome per task from the Phase 3 cache.
    always_correct = []
    missing = 0

    for qid in qids:
        key = f"task_{qid}_retr"

        if key in p3_cache:
            always_correct.append(bool(p3_cache[key]["correct"]))
        else:
            missing += 1
            always_correct.append(False)

    if missing:
        print(f"[warn] {model}: {missing}/{n} tasks missing from phase3 cache")

    init_acc, init_lo, init_hi = bootstrap_ci(
        initial_correct,
        lambda v: float(np.mean(v))
    )

    alw_acc, alw_lo, alw_hi = bootstrap_ci(
        always_correct,
        lambda v: float(np.mean(v))
    )

    print(f"\n=== {model} (n={n}) ===")
    print(f"  no-retrieve     {init_acc:.3f} [{init_lo:.3f}, {init_hi:.3f}]")
    print(f"  always-retrieve {alw_acc:.3f} [{alw_lo:.3f}, {alw_hi:.3f}]")
    print()

    print(f"  {'arm':<22} {'f1 [95% CI]':<24} {'final acc [95% CI]':<24} {'save%':>6}")
    print(f"  {'-'*22} {'-'*24} {'-'*24} {'-'*6}")

    arms_summary = {}

    for arm_name, feats in ARMS_FINAL.items():

        if not feats:
            preds = np.zeros(n, dtype=int)
        else:
            try:
                preds, _ = loo_predictions(
                    features_per_task,
                    initial_correct,
                    feats
                )
            except (KeyError, ValueError) as e:
                print(f"  {arm_name:<22} [skipped: {e}]")
                continue

        gold = np.array([0 if c else 1 for c in initial_correct])
        pairs = list(zip(preds.tolist(), gold.tolist()))

        f1, f1_lo, f1_hi = bootstrap_ci(pairs, f1_from_pairs)

        final_per_task = [
            (always_correct[i] if preds[i] == 1 else initial_correct[i])
            for i in range(n)
        ]

        acc, acc_lo, acc_hi = bootstrap_ci(
            final_per_task,
            lambda v: float(np.mean(v))
        )

        used = int(preds.sum())
        save_rate = 1 - used / n

        f1_str = f"{f1:.3f} [{f1_lo:.3f}, {f1_hi:.3f}]"
        acc_str = f"{acc:.3f} [{acc_lo:.3f}, {acc_hi:.3f}]"

        print(
            f"  {arm_name:<22} {f1_str:<24} {acc_str:<24} {save_rate*100:>5.1f}%"
        )

        arms_summary[arm_name] = {
            "f1": f1,
            "f1_ci": [f1_lo, f1_hi],
            "final_accuracy": acc,
            "final_accuracy_ci": [acc_lo, acc_hi],
            "n_retrievals_used": used,
            "n_retrievals_saved": n - used,
            "retrieval_save_rate": save_rate,
        }

    final_summary[model] = {
        "n_tasks": n,
        "initial_accuracy": init_acc,
        "initial_accuracy_ci": [init_lo, init_hi],
        "always_retrieve_accuracy": alw_acc,
        "always_retrieve_accuracy_ci": [alw_lo, alw_hi],
        "arms": arms_summary,
    }


# Save the final paper-ready numbers
out_path = Path(OUT_DIR) / "v3_final_with_nli_and_ci.json"

with open(out_path, "w") as f:
    json.dump(final_summary, f, indent=2)

print(f"\n[saved] paper-ready numbers at {out_path}")


# Cross-model summary table
print("\n" + "=" * 90)
print("CROSS-MODEL FINAL ACCURACY (with NLI-SE) + 95% CI + retrieval-save rate")
print("=" * 90)

header = f"{'arm':<22}"

for m in final_summary:
    header += f" {m.split('/')[-1][:24]:<26}"

print(header)
print("-" * len(header))

for arm in ARMS_FINAL:
    row = f"{arm:<22}"

    for m, d in final_summary.items():
        a = d["arms"].get(arm)

        if not a:
            row += f" {'—':<26}"
            continue

        s = (
            f"{a['final_accuracy']:.3f}±"
            f"{(a['final_accuracy_ci'][1] - a['final_accuracy_ci'][0]) / 2:.3f}"
            f" ({a['retrieval_save_rate']*100:.0f}%)"
        )

        row += f" {s:<26}"

    print(row)

print("\nInterpretation:")
print("  - Compare each arm's final-acc CI to always-retrieve CI to see Pareto position.")
print("  - Compare nli_se_only vs flare_only across models to see the corrected ranking.")


=== Qwen/Qwen2.5-Coder-7B-Instruct (n=80) ===
  no-retrieve     0.863 [0.775, 0.925]
  always-retrieve 0.963 [0.912, 1.000]

  arm                    f1 [95% CI]              final acc [95% CI]        save%
  ---------------------- ------------------------ ------------------------ ------
  no_signal_baseline     0.000 [0.000, 0.000]     0.863 [0.775, 0.925]     100.0%
  saplma_only            0.273 [0.103, 0.435]     0.900 [0.825, 0.963]      58.8%
  flare_only             0.400 [0.213, 0.571]     0.950 [0.900, 0.988]      57.5%
  v6_full                0.238 [0.059, 0.400]     0.900 [0.838, 0.963]      61.3%
  cce_only               0.278 [0.074, 0.471]     0.900 [0.825, 0.963]      68.8%
  embedding_se_only      0.182 [0.000, 0.359]     0.887 [0.812, 0.950]      72.5%
  nli_se_only            0.516 [0.296, 0.700]     0.950 [0.900, 0.988]      75.0%
  flare_plus_emb_se      0.356 [0.171, 0.531]     0.925 [0.863, 0.975]      57.5%
  flare_plus_nli_se      0.500 [0.286, 0.690]     0.95